# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para la elaboración de las celdas Markdown.

Material educativo y demostrativo, sujeto a revisión y validación en Microsoft Fabric.

# NB_05 — Traducción automática con `ai.translate`

**Serie:** Microsoft Fabric AI Functions con PySpark  
**Modelo de referencia:** `gpt-5-mini`  
**Entorno:** Microsoft Fabric Notebook (PySpark)

Traduciremos solicitudes ficticias de soporte a español y portugués, revisando errores, tokens y consumo estimado.

**Referencia oficial:** [Use ai.translate with PySpark — Microsoft Learn](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/translate).

## 1. Requisitos y consideraciones

Antes de ejecutar este notebook, verifica que:

- El workspace esté asignado a una capacidad Fabric compatible.
- El administrador haya habilitado la configuración de tenant necesaria para AI Functions.
- El notebook utilice un Fabric Runtime compatible con AI Functions.
- La sesión sea de tipo PySpark.

`ai.translate()` recibe una columna de texto y una o más etiquetas que representan los campos o entidades que se desean identificar. La función devuelve un nuevo DataFrame con columnas adicionales.

En PySpark, la transformación se distribuye a través de las particiones de Spark.

## 2. Configurar el modelo y las librerías

Utilizamos el endpoint integrado de Fabric. En Runtime 2.0 (Spark 4.1), consultar la preparación temporal indicada en [configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration).

In [1]:
from synapse.ml.spark import aifunc

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

from datetime import datetime, timezone
import uuid

MODEL_NAME = "gpt-5-mini"

aifunc.default_conf.set_deployment_name(MODEL_NAME)

print(f"Modelo configurado: {MODEL_NAME}")

StatementMeta(, bd3763d1-8735-4e2e-99ea-e927103bb222, 3, Finished, Available, Finished, False)

Modelo configurado: gpt-5-mini


## 3. Referencia del método publicado por Microsoft

### Sintaxis documentada

```python
df.ai.translate(to_lang="spanish", input_col="text", output_col="translations")
```

Fuente: [Microsoft Learn — ai.translate](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/translate).

| Parámetro | Tipo | Obligatorio | Descripción |
|---|---|---|---|
| `to_lang` | str | Sí | Idioma de destino |
| `input_col` | str | Sí | Columna de texto de origen |
| `output_col` | str | No | Columna de traducción; nombre automático si se omite |
| `error_col` | str | No | Columna de errores; nombre automático si se omite |
| `input_col_type` | str | No | Usar `path` para archivos multimodales |

Retorna un Spark DataFrame con la traducción por fila. Una entrada nula produce una salida nula. No requiere declarar un idioma de origen. No confundir estas opciones con las de configuración general del modelo.

## 4. Crear datos ficticios

Los textos representan incidentes en distintos idiomas. Incluimos un nulo para observar el comportamiento documentado. Los ejemplos siguientes son propios, no una copia del ejemplo oficial.

In [2]:
data = [(1, "The payment service is unavailable."),
        (2, "Le tableau de bord ne se met pas à jour."),
        (3, "O acesso ao sistema foi restaurado."),
        (4, "The pipeline failed after the schema change."),
        (5, None)]
df = spark.createDataFrame(data, "id_incidente INT, texto STRING")
display(df)

StatementMeta(, bd3763d1-8735-4e2e-99ea-e927103bb222, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dace0c28-d20e-4be9-8a8b-45e6a432bbdd)

## 5. Traducción básica al español

Materializamos y conservamos el resultado para reducir el riesgo de repetir inferencias en acciones Spark posteriores. Los reintentos o la pérdida de caché pueden generar nuevas llamadas.

In [3]:
from pyspark import StorageLevel

run_id = str(uuid.uuid4())
resultado = df.ai.translate(to_lang="spanish", input_col="texto",
    output_col="traduccion", error_col="error_traduccion")
resultado.persist(StorageLevel.MEMORY_AND_DISK)
resultado.count()
display(resultado)

StatementMeta(, bd3763d1-8735-4e2e-99ea-e927103bb222, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 55621389-bbae-4ae7-9f58-bea113f8ed1b)

## 6. Validar las traducciones

Revisar manualmente la fidelidad semántica y la terminología técnica. La ausencia de errores no garantiza exactitud lingüística.

In [4]:
resultado.printSchema()
display(resultado.withColumn("estado", F.when(F.col("error_traduccion").isNotNull(), "ERROR")
    .when(F.col("texto").isNull(), "ENTRADA_NULA")
    .when(F.col("traduccion").isNull(), "SIN_RESULTADO").otherwise("TRADUCIDO"))
    .groupBy("estado").count())

StatementMeta(, bd3763d1-8735-4e2e-99ea-e927103bb222, 6, Finished, Available, Finished, False)

root
 |-- id_incidente: integer (nullable = true)
 |-- texto: string (nullable = true)
 |-- traduccion: string (nullable = true)
 |-- error_traduccion: struct (nullable = true)
 |    |-- response: string (nullable = true)
 |    |-- status: struct (nullable = true)
 |    |    |-- protocolVersion: struct (nullable = true)
 |    |    |    |-- protocol: string (nullable = true)
 |    |    |    |-- major: integer (nullable = true)
 |    |    |    |-- minor: integer (nullable = true)
 |    |    |-- statusCode: integer (nullable = true)
 |    |    |-- reasonPhrase: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 9dcbe73e-8494-4af8-9342-c391e1e09fce)

## 8. Consultar estadísticas con `ai.stats`

Las estadísticas se consultan desde el DataFrame resultante de la operación:

```python
stats_df = resultado.ai.stats
display(stats_df)
```

La documentación incluye métricas de ejecución y consumo de tokens, entre ellas `num_successful`, `num_exceptions`, `num_unevaluated`, `input_tokens`, `cached_tokens`, `output_tokens` y `reasoning_tokens`.

La disponibilidad de cada métrica depende de la respuesta del servicio. Por eso, el código consulta los campos de forma defensiva.

In [5]:
stats_df = resultado.ai.stats

display(stats_df)

stats_rows = stats_df.collect()

if not stats_rows:
    raise RuntimeError(
        "AI Functions no devolvió estadísticas de ejecución."
    )

stats = stats_rows[0].asDict(recursive=True)

print("===== ESTADÍSTICAS AI TRANSLATE =====")
for key, value in stats.items():
    print(f"{key}: {value}")

StatementMeta(, bd3763d1-8735-4e2e-99ea-e927103bb222, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4651e8c8-b892-48b7-af75-36566ecc3cc9)

===== ESTADÍSTICAS AI TRANSLATE =====
num_successful: 4
num_exceptions: 0
num_unevaluated: 1
cached_tokens: 0
input_tokens: 753
output_tokens: 385
reasoning_tokens: 192
client_type: fabric_llm_endpoint
model: gpt-5-mini


## 9. Estimar el consumo de capacidad

Para `gpt-5-mini`, utilizaremos las tarifas publicadas para el endpoint integrado de Fabric:

| Tipo de token | CU-segundos por 1.000 tokens |
|---|---:|
| Entrada | 8,40 |
| Entrada en caché | 0,84 |
| Salida | 67,23 |

La fórmula es:

$$
CU_s = \\frac{(I-C)\\times 8.40 + C\\times 0.84 + O\\times 67.23}{1000}
$$

Donde:

- \(I\): tokens de entrada.
- \(C\): tokens de entrada en caché.
- \(O\): tokens de salida.

**Alcance:** el cálculo es una estimación del consumo de las llamadas al modelo integrado. No representa el consumo total de Spark ni el consumo completo de capacidad del workspace.

Los tokens de razonamiento se muestran como dato informativo y no se agregan por separado a la fórmula.

In [6]:
input_tokens = int(stats.get("input_tokens") or 0)
cached_tokens = int(stats.get("cached_tokens") or 0)
output_tokens = int(stats.get("output_tokens") or 0)
reasoning_tokens = int(stats.get("reasoning_tokens") or 0)

if cached_tokens > input_tokens:
    raise ValueError(
        "cached_tokens no puede ser mayor que input_tokens."
    )

INPUT_RATE = 8.40
CACHED_RATE = 0.84
OUTPUT_RATE = 67.23

regular_input_tokens = input_tokens - cached_tokens

estimated_cu_seconds = (
    regular_input_tokens * INPUT_RATE
    + cached_tokens * CACHED_RATE
    + output_tokens * OUTPUT_RATE
) / 1000

print("===== ESTIMACIÓN DE CONSUMO =====")
print(f"Modelo reportado: {stats.get('model', MODEL_NAME)}")
print(f"Tokens de entrada: {input_tokens:,}")
print(f"Tokens en caché: {cached_tokens:,}")
print(f"Tokens de entrada normales: {regular_input_tokens:,}")
print(f"Tokens de salida: {output_tokens:,}")
print(f"Tokens de razonamiento (informativo): {reasoning_tokens:,}")
print(f"CU-segundos estimados: {estimated_cu_seconds:.6f}")

StatementMeta(, bd3763d1-8735-4e2e-99ea-e927103bb222, 8, Finished, Available, Finished, False)

===== ESTIMACIÓN DE CONSUMO =====
Modelo reportado: gpt-5-mini
Tokens de entrada: 753
Tokens en caché: 0
Tokens de entrada normales: 753
Tokens de salida: 385
Tokens de razonamiento (informativo): 192
CU-segundos estimados: 32.208750


## 9. Construir un registro de auditoría

Se audita la ejecución al español. El ejemplo opcional en portugués queda fuera de este registro. Las filas nulas se consideran esperadas y no requieren traducción.

In [7]:
def metric_int(name):
    value = stats.get(name)
    return int(value) if value is not None else 0

num_input_rows = df.count()
num_successful = metric_int("num_successful")
num_exceptions = metric_int("num_exceptions")
num_unevaluated = metric_int("num_unevaluated")
num_harmful = metric_int("num_harmful")

if num_exceptions > 0 or num_harmful > 0:
    execution_status = "PARTIAL"
elif num_successful == df.filter(F.col("texto").isNotNull()).count():
    execution_status = "SUCCESS"
else:
    execution_status = "INCOMPLETE"

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp_utc", StringType(), False),
    StructField("notebook_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])

audit_record = [(
    run_id,
    datetime.now(timezone.utc).isoformat(),
    "NB_05_AI_TRANSLATE",
    "ai.translate",
    MODEL_NAME,
    str(stats.get("model") or MODEL_NAME),
    num_input_rows,
    num_successful,
    num_exceptions,
    num_unevaluated,
    num_harmful,
    input_tokens,
    cached_tokens,
    output_tokens,
    reasoning_tokens,
    float(estimated_cu_seconds),
    str(stats.get("client_type") or "unknown"),
    execution_status
)]

audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)

display(audit_df)

StatementMeta(, bd3763d1-8735-4e2e-99ea-e927103bb222, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 25178d96-0228-4db1-af2b-f4b3e43901cd)

## 11. Persistencia opcional en Lakehouse

Si deseas almacenar la auditoría en una tabla Delta del Lakehouse adjunto al notebook, puedes habilitar esta celda.

El modo `append` agrega un registro por ejecución. Verifica previamente el Lakehouse y el nombre de tabla.

In [ ]:
# Descomenta para persistir la auditoría en el Lakehouse adjunto.

# TABLE_NAME = "dbo.audit_ai_functions"
#
# (
#     audit_df.write
#     .format("delta")
#     .mode("append")
#     .saveAsTable(TABLE_NAME)
# )
#
# print(f"Auditoría guardada: {TABLE_NAME}")

## 11. Ejemplo avanzado: otro idioma de destino

Ejemplo opcional que genera consumo adicional. Ejecutar solo si deseas probar una segunda traducción. Sus estadísticas se muestran por separado y no están incluidas en la auditoría anterior.

In [8]:
RUN_PORTUGUESE = False
if RUN_PORTUGUESE:
    resultado_pt = df.ai.translate(to_lang="portuguese", input_col="texto",
        output_col="traducao", error_col="erro").persist(StorageLevel.MEMORY_AND_DISK)
    resultado_pt.count()
    display(resultado_pt)
    display(resultado_pt.ai.stats)
    resultado_pt.unpersist()

StatementMeta(, bd3763d1-8735-4e2e-99ea-e927103bb222, 10, Finished, Available, Finished, False)

## 12. Extensión multimodal

La documentación oficial admite rutas a archivos mediante `input_col_type="path"`. Requiere configuración y permisos adecuados. Ejemplo no ejecutado por defecto:

```python
resultado_archivos = df_archivos.ai.translate(
    to_lang="spanish", input_col="ruta", input_col_type="path",
    output_col="traduccion_documento", error_col="error")
```

[Configuración multimodal](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview).

## 13. Liberar caché

In [9]:
resultado.unpersist()

StatementMeta(, bd3763d1-8735-4e2e-99ea-e927103bb222, 11, Finished, Available, Finished, True)

DataFrame[id_incidente: int, texto: string, traduccion: string, error_traduccion: struct<response:string,status:struct<protocolVersion:struct<protocol:string,major:int,minor:int>,statusCode:int,reasonPhrase:string>>]

## 14. Conclusiones y referencias

Se realizó una demostración de traducción por fila, revisión de errores y estimación de consumo. Las tarifas de referencia deben verificarse antes de presupuestar. La estimación no incluye el consumo Spark ni equivale a la medición total de capacidad.

- [Método ai.translate](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/translate)
- [Configuración y estadísticas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)

Validación pendiente en una capacidad real de Fabric.